# 01 - Preparacion del dataset


## Aviso inicial

Este notebook construye `data/processed/` a partir de `data/raw/`. El analisis previo detecto que los datasets estan actualmente comprimidos en `Images/`:

- `D-Fire.zip` contiene YOLO `.txt`.
- `day_time_wildfire_v2.tar.gz` contiene imagenes y Pascal VOC `.xml` para humo.
- `cloud.tar.gz` contiene imagenes negativas sin anotacion.

Extraelos primero en `data/raw/dfire/`, `data/raw/wildfire_smoke/` y `data/raw/cloud_fog/`. Si falta algun dataset, la seleccion fallara con un mensaje indicando cuantas imagenes/anotaciones se han encontrado y cuantas hacen falta.


## 1. Configuracion y utilidades


In [ ]:
from pathlib import Path
import json
import random
import shutil
import sys
import time
import warnings
import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATASETS = {
    "dfire": RAW_DIR / "dfire",
    "wildfire_smoke": RAW_DIR / "wildfire_smoke",
    "cloud_fog": RAW_DIR / "cloud_fog",
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}
CLASS_NAMES = {0: "fire", 1: "smoke"}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Project root: {PROJECT_ROOT}")


def find_images(root: Path):
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS and not p.name.startswith("._"))


def detect_annotation_formats(root: Path):
    if not root.exists():
        return ["missing"]
    formats = []
    if list(root.rglob("*.txt")):
        formats.append("YOLO .txt or text")
    if list(root.rglob("*.xml")):
        formats.append("Pascal VOC .xml")
    json_files = list(root.rglob("*.json"))
    if json_files:
        coco = False
        for path in json_files[:5]:
            try:
                data = json.loads(path.read_text(encoding="utf-8"))
                coco = coco or {"images", "annotations"}.issubset(data.keys())
            except Exception:
                pass
        formats.append("COCO .json" if coco else "JSON")
    return formats or ["none detected"]


def sibling_yolo_label(image_path: Path):
    candidates = [image_path.with_suffix(".txt")]
    parts = list(image_path.parts)
    if "images" in parts:
        idx = parts.index("images")
        label_parts = parts.copy()
        label_parts[idx] = "labels"
        candidates.append(Path(*label_parts).with_suffix(".txt"))
    return next((p for p in candidates if p.exists()), None)


def parse_yolo_lines(label_path: Path, allowed_classes={0, 1}):
    boxes = []
    if not label_path or not label_path.exists() or label_path.stat().st_size == 0:
        return boxes
    for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
        except ValueError:
            continue
        if cls in allowed_classes and all(0 <= v <= 1 for v in (x, y, w, h)) and w > 0 and h > 0:
            boxes.append((cls, x, y, w, h))
    return boxes


def yolo_to_xyxy(box, width, height):
    cls, x, y, w, h = box
    x1 = int((x - w / 2) * width)
    y1 = int((y - h / 2) * height)
    x2 = int((x + w / 2) * width)
    y2 = int((y + h / 2) * height)
    return cls, max(0, x1), max(0, y1), min(width - 1, x2), min(height - 1, y2)


def draw_yolo_boxes(ax, image_path: Path, boxes):
    img = Image.open(image_path).convert("RGB")
    width, height = img.size
    ax.imshow(img)
    for box in boxes:
        cls, x1, y1, x2, y2 = yolo_to_xyxy(box, width, height)
        color = "orangered" if cls == 0 else "gray"
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2, color=color))
        ax.text(x1, max(0, y1 - 4), CLASS_NAMES.get(cls, str(cls)), color="white", fontsize=8,
                bbox=dict(facecolor=color, alpha=0.8, edgecolor="none", pad=1))
    ax.set_title(image_path.name[:32], fontsize=8)
    ax.axis("off")


from sklearn.model_selection import train_test_split
import yaml


def detect_split(path: Path):
    parts = {p.lower() for p in path.parts}
    for split in ("train", "val", "valid", "validation", "test"):
        if split in parts:
            return "val" if split in {"valid", "validation"} else split
    return "unspecified"


def yolo_lines_from_boxes(boxes):
    return [f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for cls, x, y, w, h in boxes]


def voc_boxes_for_image(image_path: Path, class_id=1):
    candidates = [image_path.with_suffix(".xml")]
    for parent in image_path.parents:
        candidates.extend([
            parent / "Annotations" / f"{image_path.stem}.xml",
            parent / "annotations" / f"{image_path.stem}.xml",
            parent / "annotations" / "xmls" / f"{image_path.stem}.xml",
        ])
    xml_path = next((p for p in candidates if p.exists()), None)
    if not xml_path:
        return []
    try:
        root = ET.parse(xml_path).getroot()
        width = int(root.findtext("size/width") or Image.open(image_path).size[0])
        height = int(root.findtext("size/height") or Image.open(image_path).size[1])
        boxes = []
        for obj in root.findall("object"):
            name = (obj.findtext("name") or "smoke").lower()
            if "smoke" not in name and name not in {"object", "1"}:
                continue
            b = obj.find("bndbox")
            xmin, ymin = float(b.findtext("xmin")), float(b.findtext("ymin"))
            xmax, ymax = float(b.findtext("xmax")), float(b.findtext("ymax"))
            boxes.append((class_id, ((xmin + xmax) / 2) / width, ((ymin + ymax) / 2) / height,
                          (xmax - xmin) / width, (ymax - ymin) / height))
        return [b for b in boxes if b[3] > 0 and b[4] > 0]
    except Exception as exc:
        print(f"[WARN] VOC invalido {xml_path}: {exc}")
        return []


def load_coco_maps(root: Path, class_id=1):
    maps = {}
    for json_path in root.rglob("*.json"):
        try:
            data = json.loads(json_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not {"images", "annotations"}.issubset(data.keys()):
            continue
        images = {img["id"]: img for img in data.get("images", [])}
        cats = {cat["id"]: str(cat.get("name", "")).lower() for cat in data.get("categories", [])}
        for ann in data.get("annotations", []):
            cat_name = cats.get(ann.get("category_id"), "smoke")
            if cats and "smoke" not in cat_name:
                continue
            img = images.get(ann.get("image_id"))
            if not img or "bbox" not in ann:
                continue
            x, y, w, h = ann["bbox"]
            iw, ih = img.get("width"), img.get("height")
            if iw and ih and w > 0 and h > 0:
                maps.setdefault(Path(img["file_name"]).name, []).append(
                    (class_id, (x + w / 2) / iw, (y + h / 2) / ih, w / iw, h / ih)
                )
    return maps


def wildfire_boxes(image_path: Path, coco_map):
    # ADAPTADO: AI For Mankind suele venir en Pascal VOC; tambien aceptamos YOLO y COCO.
    yolo = parse_yolo_lines(sibling_yolo_label(image_path))
    if yolo:
        return [(1, x, y, w, h) for _, x, y, w, h in yolo]
    voc = voc_boxes_for_image(image_path, class_id=1)
    return voc if voc else coco_map.get(image_path.name, [])


# ADAPTADO: el D-Fire.zip local declara names: ['smoke', 'fire'].
# El dataset final del proyecto debe ser 0=fire y 1=smoke, asi que se remapea.
DFIRE_CLASS_REMAP = {0: 1, 1: 0}


def parse_dfire_yolo_lines(label_path: Path):
    return [(DFIRE_CLASS_REMAP.get(cls, cls), x, y, w, h) for cls, x, y, w, h in parse_yolo_lines(label_path)]


## 2. Seleccion de imagenes


In [ ]:
def collect_dfire_candidates():
    candidates = []
    for image_path in find_images(DATASETS["dfire"]):
        boxes = parse_dfire_yolo_lines(sibling_yolo_label(image_path))
        classes = {cls for cls, *_ in boxes}
        if boxes and classes.intersection({0, 1}):
            candidates.append({
                "block": "dfire", "image": image_path, "boxes": boxes, "classes": classes,
                "strata": "fire_smoke" if classes == {0, 1} else ("fire" if 0 in classes else "smoke"),
                "source_split": detect_split(image_path),
            })
    return candidates


def allocate_counts(total, groups):
    raw = {k: total * v / sum(groups.values()) for k, v in groups.items()}
    counts = {k: int(v) for k, v in raw.items()}
    remainder = total - sum(counts.values())
    order = sorted(raw, key=lambda k: raw[k] - counts[k], reverse=True)
    for key in order[:remainder]:
        counts[key] += 1
    return counts


def balanced_dfire_selection(candidates, total=500):
    rng = random.Random(RANDOM_SEED)
    if len(candidates) < total:
        raise ValueError(f"D-Fire tiene {len(candidates)} candidatas validas; se necesitan {total}.")
    by_split = {}
    for item in candidates:
        by_split.setdefault(item["source_split"], []).append(item)
    targets = allocate_counts(total, {k: len(v) for k, v in by_split.items()})
    selected, selected_images = [], set()
    for split, split_target in targets.items():
        pool = by_split[split][:]
        local, local_images = [], set()
        fire_target = split_target // 2
        smoke_target = split_target - fire_target
        for cls, preferred_strata, target in [(0, "fire", fire_target), (1, "smoke", smoke_target)]:
            preferred_pool = [c for c in pool if c["strata"] == preferred_strata]
            fallback_pool = [c for c in pool if cls in c["classes"] and c["strata"] != preferred_strata]
            rng.shuffle(preferred_pool)
            rng.shuffle(fallback_pool)
            for item in preferred_pool + fallback_pool:
                if sum(cls in s["classes"] for s in local) >= target:
                    break
                if item["image"] not in selected_images and item["image"] not in local_images:
                    local.append(item)
                    local_images.add(item["image"])
        rest = [c for c in pool if c["image"] not in selected_images and c["image"] not in local_images]
        rng.shuffle(rest)
        local.extend(rest[: max(0, split_target - len(local))])
        for item in local[:split_target]:
            selected.append(item)
            selected_images.add(item["image"])
    rest = [c for c in candidates if c["image"] not in selected_images]
    rng.shuffle(rest)
    selected.extend(rest[: max(0, total - len(selected))])
    return selected[:total]


def collect_wildfire_candidates(total=700):
    coco_map = load_coco_maps(DATASETS["wildfire_smoke"], class_id=1)
    candidates = []
    for image_path in find_images(DATASETS["wildfire_smoke"]):
        boxes = wildfire_boxes(image_path, coco_map)
        if boxes:
            candidates.append({"block": "wildfire_smoke", "image": image_path, "boxes": boxes,
                               "classes": {1}, "strata": "smoke", "source_split": detect_split(image_path)})
    if len(candidates) < total:
        raise ValueError(f"Wildfire Smoke tiene {len(candidates)} candidatas con humo; se necesitan {total}.")
    random.Random(RANDOM_SEED).shuffle(candidates)
    return candidates[:total]


def collect_cloud_fog_candidates(total=300):
    images = find_images(DATASETS["cloud_fog"])
    if len(images) < total:
        raise ValueError(f"Cloud/Fog tiene {len(images)} imagenes; se necesitan {total}.")
    random.Random(RANDOM_SEED).shuffle(images)
    return [{"block": "cloud_fog", "image": p, "boxes": [], "classes": set(),
             "strata": "background", "source_split": detect_split(p)} for p in images[:total]]


for name, path in DATASETS.items():
    print(f"{name}: {len(find_images(path))} imagenes, formatos: {detect_annotation_formats(path)}")

selected = []
selected.extend(balanced_dfire_selection(collect_dfire_candidates(), 700))
selected.extend(collect_wildfire_candidates(700))
selected.extend(collect_cloud_fog_candidates(300))
print(f"Total seleccionado: {len(selected)}")
display(pd.DataFrame([{"block": s["block"], "strata": s["strata"], "source_split": s["source_split"]} for s in selected])
        .groupby(["block", "strata", "source_split"]).size().rename("imagenes").reset_index())


## 3. Division estratificada


In [ ]:
def safe_split(items, test_size, labels, seed):
    counts = pd.Series(labels).value_counts()
    stratify = labels if len(counts) > 1 and counts.min() >= 2 else None
    if stratify is None:
        warnings.warn("No hay suficientes muestras por estrato; se divide sin stratify.")
    return train_test_split(items, test_size=test_size, random_state=seed, stratify=stratify)

labels = [item["strata"] for item in selected]
train_items, temp_items = safe_split(selected, 0.30, labels, RANDOM_SEED)
val_items, test_items = safe_split(temp_items, 0.50, [item["strata"] for item in temp_items], RANDOM_SEED)
splits = {"train": train_items, "val": val_items, "test": test_items}
for split, items in splits.items():
    print(split, len(items), pd.Series([i["strata"] for i in items]).value_counts().to_dict())


## 4. Copia y dataset.yaml


In [ ]:
def reset_processed_dirs():
    for split in ("train", "val", "test"):
        for kind in ("images", "labels"):
            target = PROCESSED_DIR / kind / split
            if target.exists():
                shutil.rmtree(target)
            target.mkdir(parents=True, exist_ok=True)


def copy_split(split_name, items):
    image_dir = PROCESSED_DIR / "images" / split_name
    label_dir = PROCESSED_DIR / "labels" / split_name
    for idx, item in enumerate(items):
        src = item["image"]
        out_name = f"{item['block']}_{idx:04d}_{src.stem}{src.suffix.lower()}"
        dst_img = image_dir / out_name
        dst_lbl = label_dir / f"{Path(out_name).stem}.txt"
        shutil.copy2(src, dst_img)
        text = "\n".join(yolo_lines_from_boxes(item["boxes"]))
        dst_lbl.write_text(text + ("\n" if text else ""), encoding="utf-8")

reset_processed_dirs()
for split, items in splits.items():
    copy_split(split, items)

dataset_yaml = PROJECT_ROOT / "data" / "dataset.yaml"
yaml_data = {
    "path": str(PROCESSED_DIR.resolve()).replace("\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 2,
    "names": {0: "fire", 1: "smoke"},
}
dataset_yaml.write_text(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(dataset_yaml.read_text(encoding="utf-8"))


## 5. Validacion del dataset procesado


In [ ]:
rows, missing = [], []
for split in ("train", "val", "test"):
    images = find_images(PROCESSED_DIR / "images" / split)
    labels = sorted((PROCESSED_DIR / "labels" / split).glob("*.txt"))
    rows.append({"split": split, "images": len(images), "labels": len(labels)})
    label_stems = {p.stem for p in labels}
    missing.extend([p for p in images if p.stem not in label_stems])
validation = pd.DataFrame(rows)
display(validation)
assert validation.set_index("split").loc["train", "images"] == 1190
assert validation.set_index("split").loc["val", "images"] == 255
assert validation.set_index("split").loc["test", "images"] == 255
assert not missing, f"Imagenes sin etiqueta: {missing[:5]}"

train_images = find_images(PROCESSED_DIR / "images" / "train")
sample = random.sample(train_images, min(9, len(train_images)))
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for ax, image_path in zip(axes.ravel(), sample):
    label = PROCESSED_DIR / "labels" / "train" / f"{image_path.stem}.txt"
    draw_yolo_boxes(ax, image_path, parse_yolo_lines(label))
for ax in axes.ravel()[len(sample):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

class_rows = []
for split in ("train", "val", "test"):
    counts = {"fire": 0, "smoke": 0, "background": 0}
    for label_path in (PROCESSED_DIR / "labels" / split).glob("*.txt"):
        boxes = parse_yolo_lines(label_path)
        if not boxes:
            counts["background"] += 1
        else:
            present = {cls for cls, *_ in boxes}
            counts["fire"] += int(0 in present)
            counts["smoke"] += int(1 in present)
    class_rows.append({"split": split, **counts})
class_df = pd.DataFrame(class_rows).set_index("split")
display(class_df)
class_df.plot(kind="bar", stacked=True, figsize=(8, 4), title="Distribucion por split")
plt.ylabel("imagenes con presencia de clase")
plt.tight_layout()
plt.show()
